# 09 · ¿Cambia el resultado de las reglas según el entorno?

Estudiamos por separado momentum y reversión: ¿se relaciona una característica conocida antes de decidir con el resultado del mes siguiente? Este es un **cribado descriptivo del desarrollo 1927–1989**, no una evaluación fuera de muestra ni una selección definitiva de variables.

Antes de empezar, ejecuta **01 → 02 → 07 → 08**, en ese orden. Selecciona `.venv` como kernel y ejecuta las celdas de este cuaderno en orden. No se descargan fuentes.

Las hipótesis y el procedimiento se describen en [Hipótesis y protocolo](../docs/hipotesis_y_cribado.md). Ya se había explorado el desarrollo: no se presenta el protocolo actual como anterior a toda consulta de datos. La ausencia de asociaciones robustas también es un resultado válido.

## 1. Entradas y decisiones fijadas

Utilizamos las diez características y los supuestos contables de las etiquetas, pero **no usamos las etiquetas como respuesta**. Los modelos llegarán después. La comparación principal incluye costes y dividendos, no se elige aquí la convención que dé mejores resultados.

Haremos veinte regresiones: diez características por dos reglas. Se esperan pendientes positivas para momentum en su indicador 12–1, retorno de 24 meses, distancia a la media, drawdown más cercano a cero y amplitud. Las demás relaciones son exploratorias, sin un signo previo. Las hipótesis son adaptaciones a nuestro índice, no conclusiones ya demostradas por los artículos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import project_root, read_result, save_result
from tfm.features import FEATURE_COLUMNS, feature_dictionary
from tfm.screening import (MIN_HISTORY, HAC_LAGS, BH_ALPHA, MIN_SHIFT,
                          build_screening_monthly, screen_relationships, describe_regimes)

ROOT = project_root()
caracteristicas, _ = read_result(ROOT, "monthly_features")
componentes, _ = read_result(ROOT, "monthly_return_components")
senales, _ = read_result(ROOT, "shiller_signals")
etiquetas, informe_etiquetas = read_result(ROOT, "labelled_signals")
parametros = {"initial_capital": 100.0, "transaction_cost_rate": 0.001,
              "annual_borrow_rate": 0.01, "include_dividends": True}
if informe_etiquetas["parameters"] != parametros:
    raise ValueError("Utiliza los mismos supuestos centrales en los notebooks 08 y 09.")
nombres = feature_dictionary().set_index("feature").description.to_dict()
reglas = {"momentum": "Momentum", "reversal": "Reversión"}
display(pd.Series({"Historia anterior mínima (meses)": MIN_HISTORY,
                   "Retardos Newey–West (meses)": HAC_LAGS,
                   "BH: p ajustado <": BH_ALPHA,
                   "Desplazamiento circular mínimo (meses)": MIN_SHIFT}, name="Protocolo"))
display(pd.Series(parametros, name="Supuestos contables"))

## 2. Retorno de la regla y preparación con el pasado

La respuesta es **retorno neto de la regla sin filtrar menos RF del mismo mes**. Se simulan primero todos los meses, manteniendo posiciones, después se toma la muestra común. También se incluyen meses en efectivo. Así describimos la regla completa, aunque una asociación puede reflejar que opera más y no que sus operaciones sean mejores.

Para cada característica usamos al menos 36 observaciones anteriores. Con ese pasado calculamos los percentiles 1 y 99, recortamos los extremos y calculamos media y desviación típica muestral de la historia recortada. El valor actual se recorta con esos límites y se expresa en esa escala. **El mes actual no participa en los estadísticos y los retornos de las estrategias no se recortan.**

El calentamiento puede reducir la muestra respecto a los 749 meses con las variables originales completas. No se rellenan ausencias ni se comprimen huecos del calendario.

In [ ]:
mensual, informe = build_screening_monthly(caracteristicas, componentes, senales, parametros)
comun = mensual.loc[mensual.screening_ready].copy()
display(pd.DataFrame([{
    "Meses por regla antes del calentamiento": informe["calendar_months"],
    "Meses comunes por regla": informe["common_months"],
    "Primer objetivo común": informe["first_target"],
    "Último objetivo común": informe["last_target"],
    "Meses excluidos por regla": len(informe["excluded_targets"]),
}]))
display(pd.DataFrame(informe["excluded_targets"]).head(5))
print("Cada fila conserva la decisión y las fechas de fuente. La lista completa de exclusiones se guardará en el informe.")

### Un mismo mes, dos preguntas distintas

Una operación aislada abre desde efectivo y cierra al terminar el mes. La regla continua puede conservar la posición que ya tenía y pagar menos negociación. Comparamos ambas cuentas para entender la diferencia, el cribado usa exclusivamente el retorno de la **regla continua**. El resultado observado del mes objetivo nunca se incorpora como predictor.

In [ ]:
mes_objetivo = "1975-05"
if mes_objetivo not in set(mensual.target_period):
    raise ValueError("Escoge un objetivo entre 1927-01 y 1989-12.")
ejemplo = mensual.loc[mensual.target_period.eq(mes_objetivo),
                      ["strategy", "decision_period", "target_period", "position", "anomaly_return"]].merge(
    etiquetas[["strategy", "target_period", "excess_return", "label"]],
    on=["strategy", "target_period"], validate="one_to_one",
)
display(ejemplo.rename(columns={"strategy": "Regla", "decision_period": "Decisión",
    "target_period": "Objetivo", "position": "Posición",
    "anomaly_return": "Exceso de la regla continua", "excess_return": "Exceso de operación aislada",
    "label": "Etiqueta del 08"}).set_index("Regla"))

## 3. Veinte relaciones y corrección por pruebas múltiples

Ajustamos `retorno de la regla − RF = a + b × característica estandarizada + error`. La **pendiente b** describe cuánto cambia el retorno medio al aumentar una unidad en esa escala histórica. No demuestra causalidad ni que el filtro gane dinero.

Newey–West calcula un error estándar que tiene en cuenta variabilidad cambiante y dependencia temporal, con seis retardos y pesos Bartlett. Usamos su versión asintótica, sin multiplicar por `n/(n−2)`, y valores p bilaterales aproximados por una normal. No elegimos el retardo después de ver resultados.

**Benjamini–Hochberg (BH), con p ajustado < 0,10, es el único criterio de significación del cribado.** Se calcula sobre las veinte pruebas conjuntamente, no por separado para cada regla ni únicamente para las relaciones favorables.

BH busca controlar la proporción esperada de falsos descubrimientos entre los hallazgos; el 10 % no es la probabilidad de que una relación concreta sea falsa. Su garantía requiere valores p adecuados y pruebas independientes o determinadas formas de dependencia positiva, no certificadas aquí.

Las relaciones no estimables permanecen en la tabla y cuentan con p = 1 a efectos del ajuste. El signo y las sombras se muestran por separado: una pendiente negativa no respalda una hipótesis positiva aunque supere BH. Los valores p son aproximaciones bajo supuestos, no garantías.

In [ ]:
contrastes, sombras, informe_sombras = screen_relationships(mensual)
estados = {"not_estimable": "No estimable", "no_bh_evidence": "Sin respaldo tras BH",
           "bh_evidence": "Supera BH"}
direcciones = {"not_estimable": "No estimable", "exploratory": "Exploratoria, sin signo previo",
               "supports_expected_sign": "Respalda el signo esperado",
               "does_not_support_expected_sign": "No respalda el signo esperado"}
tabla = contrastes[["strategy", "feature", "expected_sign", "months", "slope", "t_hac",
                    "p_value", "p_bh", "evidence_status", "direction_status", "exceeds_shadows"]].copy()
tabla["strategy"] = tabla.strategy.map(reglas)
tabla["feature"] = tabla.feature.map(nombres)
tabla["expected_sign"] = tabla.expected_sign.replace({"unspecified": "Sin signo"})
tabla["evidence_status"] = tabla.evidence_status.map(estados)
tabla["direction_status"] = tabla.direction_status.map(direcciones)
tabla["exceeds_shadows"] = tabla.exceeds_shadows.map({True: "Sí", False: "No"}).fillna("No disponible")
with pd.option_context("display.max_rows", 30, "display.max_columns", 14):
    display(tabla.rename(columns={"strategy": "Regla", "feature": "Característica", "expected_sign": "Signo esperado",
        "months": "Meses", "slope": "Pendiente", "t_hac": "t HAC", "p_value": "p sin ajustar",
        "p_bh": "p BH (20)", "evidence_status": "Evidencia estadística",
        "direction_status": "Dirección", "exceeds_shadows": "Supera sombras"}).set_index(["Regla", "Característica"]))

## 4. Terciles y estabilidad entre periodos

Los terciles separan valores bajos, intermedios y altos con cortes calculados solo con el pasado de cada variable. Si coinciden los cortes, no se reparten valores idénticos artificialmente para llenar grupos. Mostramos cuántos meses y señales contiene cada grupo.

El Sharpe condicional usa media y desviación del exceso mensual y raíz de doce. **Resume meses que pueden estar separados, no es una estrategia ejecutable ni una curva de inversión.** Una mayor volatilidad puede cambiar el riesgo sin producir una pendiente clara en la rentabilidad media.

Los cuatro subperiodos están fijados: 1927–1942, 1943–1958, 1959–1974 y 1975–1989. Solo se usan las fechas disponibles después del calentamiento. No añadimos contrastes por tramo ni escogemos los años que dan el resultado deseado.

In [ ]:
terciles, estabilidad = describe_regimes(mensual)
# Puedes inspeccionar otra característica, no cambia el protocolo ni selecciona un modelo.
caracteristica = "market_return_24m"
if caracteristica not in FEATURE_COLUMNS:
    raise ValueError("Escoge una de las diez características.")
display(terciles.loc[terciles.feature.eq(caracteristica)].rename(columns={
    "strategy": "Regla", "tercile": "Tercil", "months": "Meses", "signal_months": "Meses con señal",
    "signal_fraction": "Fracción con señal", "mean_excess_return": "Exceso mensual medio",
    "annualized_conditional_sharpe": "Sharpe condicional anualizado"}).drop(columns="feature"))
display(estabilidad.loc[estabilidad.feature.eq(caracteristica)].drop(columns="feature"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for regla, nombre in reglas.items():
    datos = terciles.loc[terciles.feature.eq(caracteristica) & terciles.strategy.eq(regla)]
    axes[0].plot(datos.tercile, datos.mean_excess_return * 100, marker="o", label=nombre)
    datos = estabilidad.loc[estabilidad.feature.eq(caracteristica) & estabilidad.strategy.eq(regla)]
    axes[1].plot(datos.subperiod_start.str[:4], datos.slope * 100, marker="o", label=nombre)
axes[0].set(title=f"Exceso medio por tercil: {nombres[caracteristica]}", xlabel="Tercil histórico", ylabel="Exceso mensual (%)", xticks=[1, 2, 3])
axes[1].set(title="Pendiente en los cuatro tramos fijos", xlabel="Inicio del tramo", ylabel="Puntos porcentuales por unidad de z")
for ax in axes:
    ax.axhline(0, color="gray", linewidth=.8)
    ax.grid(alpha=.2)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. ¿La relación destaca frente a variables desplazadas?

Las **sombras circulares** desplazan conjuntamente las diez series estandarizadas, dejando fijos los resultados de las reglas. Se rompe su alineación temporal, conservando parte de su estructura. Solo usamos desplazamientos separados al menos 36 meses por ambos lados del círculo.

Para cada desplazamiento calculamos el mayor `abs(t HAC)` de las veinte relaciones. El umbral es el percentil 95 de esos máximos, con cuantil por orden superior. Así comparamos con una familia completa, no con la sombra que más nos convenga. Cada sombra requiere las veinte relaciones estimables. Si alguna queda incompleta, se conserva su diagnóstico y toda la comparación con sombras queda no disponible; no se calcula el umbral con una familia reducida. El ajuste BH se mantiene. Superar sombras no es un requisito para cumplir BH; es una comprobación adicional.

Es una comprobación de plausibilidad: la frontera circular es artificial y los regímenes cambian, por lo que no es una prueba exacta de azar. No demuestra capacidad predictiva fuera de muestra.

In [ ]:
display(pd.Series(informe_sombras, name="Sombras"))
resumen = contrastes.groupby(["strategy", "evidence_status"]).size().rename("Relaciones").reset_index()
resumen["strategy"] = resumen.strategy.map(reglas)
resumen["evidence_status"] = resumen.evidence_status.map(estados)
display(resumen.rename(columns={"strategy": "Regla", "evidence_status": "Evidencia estadística"}))
validas = sombras.max_abs_t_hac.dropna()
if informe_sombras["shadow_comparison_available"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(validas, bins=25, alpha=.7, label="Máximo de cada familia desplazada")
    ax.axvline(informe_sombras["shadow_threshold_95"], color="red", linestyle="--", label="Percentil 95")
    ax.set(xlabel="Máximo abs(t HAC) entre las veinte relaciones", ylabel="Desplazamientos", title="Referencia de sombras circulares")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Comparación con sombras no disponible: alguna familia está incompleta. Consulta el número de relaciones estimables por sombra y los motivos de los contrastes.")

## 6. Guardar todas las tablas y delimitar la conclusión

La última celda guarda o sustituye cinco tablas Parquet y sus informes JSON en `data/processed/`: calendario preparado, veinte contrastes, terciles, subperiodos y sombras. Conserva también resultados desfavorables, exclusiones y parámetros. Los JSON son salidas descriptivas, no configuraciones.

Este cribado no elimina automáticamente ninguna candidata. Una relación no lineal puede no aparecer en una pendiente, una asociación significativa puede reflejar activación de la regla o resultar inestable. Una lista elegida con todo este desarrollo **no podrá presentarse después como selección independiente en CPCV**. La selección que use resultados se repetirá dentro del entrenamiento, con validación interna para Clustered MDA y un bloque exterior para evaluar.

No se han entrenado modelos ni analizado el tramo desde 1990. La comparación económica del filtro se realizará después frente a las reglas sin filtrar y los comparadores. Las referencias de Newey–West y Benjamini–Hochberg para añadir a la memoria están preparadas en [el protocolo](../docs/hipotesis_y_cribado.md). Jansen aporta el esquema de organización, nuestro contraste es temporal y nuestras sombras son mensuales.

In [ ]:
informe_final = {**informe, **informe_sombras,
    "scope": "descriptive_development_screening_only",
    "protocol": "docs/hipotesis_y_cribado.md",
    "references": {"newey_west": "10.2307/1913610",
                   "benjamini_hochberg": "10.1111/j.2517-6161.1995.tb02031.x",
                   "jansen_revision": "91c903e71c1d0f1b9015f18551fadbb60bcad132"},
    "warnings": [
        "Descriptive 1927–1989 screening, no out-of-sample skill or final feature selection.",
        "Monthly Shiller means and reconstructed dividends, no executable-price interpretation.",
        "Response from continuous unfiltered rules, not isolated labels, no clipping of returns.",
        "HAC and BH rely on assumptions; BH independence or positive-dependence conditions are not certified here.",
        "BH adjusted p < 0.10 is the sole screening significance criterion; all ten candidates are retained.",
        "Circular shadows are a separate plausibility check, not an exact randomization test or a BH requirement.",
        "Tercile Sharpes describe separated months, they are not portfolio backtests.",
        "Selection based on outcomes must be repeated within training partitions, no post-1989 evaluation.",
    ],
}
salidas = {"screening_monthly": mensual, "screening_tests": contrastes,
           "screening_terciles": terciles, "screening_stability": estabilidad, "screening_shadows": sombras}
guardados = []
for nombre, datos in salidas.items():
    salida = save_result(ROOT, nombre, datos, informe_final)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Guardado en data/processed/. Cada tabla tiene su informe JSON. No se han seleccionado variables ni entrenado modelos.")